# Fine-tune Qwen3.5-4B → MITRE ATT&CK Specialist (Kaggle)

> ⚠️ **Does NOT work on Kaggle's free T4/P100** — they have no bf16, and Qwen3.5's
> GatedDeltaNet is bf16-native (fp16 is numerically broken → `BFloat16 != Half` crash;
> fp32 OOMs on 16 GB). For Qwen3.5 use **`Finetune_Colab_Qwen35.ipynb`** on a bf16 GPU
> (Colab Pro L4/A100, or a rented L4/A10). This Kaggle notebook is kept for a
> **fp16-safe base** (e.g. swap `BASE_MODEL_HF` to `Qwen/Qwen3-4B-Instruct-2507` or the
> Qwen2.5-7B legacy path in `ft_config.py`), which DO train on a free T4.

**16-bit LoRA** (NOT 4-bit). Fits a single free GPU **only for fp16-safe bases** (see warning above).

### Before you run (Kaggle right-hand panel)
1. **Settings → Accelerator → `GPU T4 x2` or `GPU P100`**.
2. **Settings → Internet → ON** (needed to pip-install + download the base model; requires a phone-verified Kaggle account).
3. **Add Input → your Dataset** containing `train.jsonl` + `val.jsonl`.
   Build them locally first and upload as a Kaggle Dataset:
   ```powershell
   cd rag_service/finetune
   python data/build_dataset.py --max-per-category 600
   # then create a Kaggle Dataset from data/output/train.jsonl + val.jsonl
   ```

Flow: install → clone repo → copy dataset → smoke test → train + export GGUF → collect.
Then on your machine: `ollama create mitre-qwen3.5:4b -f export/Modelfile.qwen35`.

In [ ]:
# 1. Install Unsloth + unsloth_zoo (pulls transformers v5, REQUIRED for Qwen3.5).
#    force-reinstall so Kaggle's preinstalled transformers is upgraded to v5.
import os
os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")  # keep CWD valid on re-runs: cell 3's `rm -rf` can delete a stale CWD → getcwd errors
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
# The protobuf bump above breaks Kaggle's preinstalled wandb (cannot import 'Imports'
# from wandb_telemetry_pb2). We don't use wandb (report_to="none"), so remove it
# rather than let trl's optional `import wandb` crash the training import.
!pip uninstall -y -q wandb
os.environ["WANDB_DISABLED"] = "true"
import transformers
print("transformers", transformers.__version__, "(need >= 5.0)")
assert int(transformers.__version__.split('.')[0]) >= 5, \
    "transformers < 5 — restart the kernel (Run → Restart) and re-run this cell."

In [ ]:
# 2. GPU sanity check
import torch
print("CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported(),
      "(T4=False → Unsloth auto-uses fp16; LoRA stays stable)")

In [ ]:
# 3. Clone the repo (for train/train_unsloth.py + ft_config.py) and cd into finetune.
#    NOTE: push your Qwen3.5 finetune changes to this branch first, or the clone
#    pulls the old trainer.
BRANCH = "main"
REPO = "https://github.com/NitithX374/CyberCase-Intelligence-Framework.git"
!rm -rf /kaggle/working/CyberCase-Intelligence-Framework
!git clone --depth 1 -b $BRANCH $REPO /kaggle/working/CyberCase-Intelligence-Framework
%cd /kaggle/working/CyberCase-Intelligence-Framework/rag_service/finetune

In [ ]:
# 4. Copy the dataset from the attached Kaggle Dataset (search /kaggle/input).
import os, shutil
os.makedirs("data/output", exist_ok=True)
found = {}
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f in ("train.jsonl", "val.jsonl"):
            found[f] = os.path.join(root, f)
print("found:", found)
assert "train.jsonl" in found, \
    "No train.jsonl under /kaggle/input — use 'Add Input' to attach your dataset."
for f, src in found.items():
    shutil.copy(src, f"data/output/{f}")
!wc -l data/output/*.jsonl

In [ ]:
# 5. Smoke test (5 steps) — confirm the loop runs before the full job.
!python train/train_unsloth.py --max-steps 5

In [ ]:
# 6. Full training (1 epoch, set in ft_config.py) + GGUF export (Q4_K_M). ~15-40 min on a T4.
#    The adapter is saved BEFORE the GGUF step, so it survives a GGUF failure.
!python train/train_unsloth.py --gguf

In [ ]:
# 7. Collect artifacts into /kaggle/working (download from the Output panel).
#    Copies the GGUF if present; always backs up the LoRA adapter so nothing is
#    lost if GGUF conversion of the new arch fails in this unsloth build.
import glob, os, shutil
ggufs = glob.glob("export/outputs/gguf/*[Qq]4_[Kk]_[Mm]*.gguf")
if ggufs:
    dst = "/kaggle/working/mitre-qwen3.5-4b-Q4_K_M.gguf"
    shutil.copy(ggufs[0], dst)
    print("GGUF:", dst, round(os.path.getsize(dst)/1e9, 2), "GB")
else:
    print("!! No GGUF produced — backing up the adapter to convert later with latest llama.cpp.")
adapter = "train/outputs/mitre-qwen-lora"
if os.path.isdir(adapter):
    shutil.make_archive("/kaggle/working/mitre-qwen3.5-lora", "zip", adapter)
    print("Adapter zip: /kaggle/working/mitre-qwen3.5-lora.zip")

## On your machine (keeps stock `qwen3.5:4b` intact)
Download `mitre-qwen3.5-4b-Q4_K_M.gguf` from the **Output** panel, then:
```powershell
cd rag_service/finetune
# put the .gguf in export/outputs/gguf/ (the Modelfile's FROM already points there)
ollama create mitre-qwen3.5:4b -f export/Modelfile.qwen35
ollama run mitre-qwen3.5:4b "What is T1059?"   # check it answers directly (no <think>)
ollama list                                     # qwen3.5:4b AND mitre-qwen3.5:4b
```
### A/B compare (base vs fine-tuned)
`ft_config.py` already defaults to `qwen3.5:4b` / `mitre-qwen3.5:4b`, so just:
```powershell
python compare/run_comparison.py --max-samples 20
# explicit override is also supported:
#   python compare/run_comparison.py --max-samples 20 --base-model qwen3.5:4b --ft-model mitre-qwen3.5:4b
```
Needs Neo4j + Qdrant up & ingested, and Ollama serving both models.